# Flow-Lenia History Dependence Offline Analysis

Этот ноутбук не запускает новые симуляции. Он читает уже готовые артефакты `experiments/frustration/history_dependence/run.sh`, строит pairwise distance matrices, late-window embedding / trajectory observables, `ΔH(w,τ)` карты, `MSC_t` summary и сохраняет результаты в отдельную директорию `results`.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur] + list(cur.parents):
        if (candidate / 'experiments').exists() and (candidate / 'scripts').exists() and (candidate / 'analysis').exists():
            return candidate
    raise RuntimeError('Could not locate repository root from current working directory.')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.history_dependence import load_analysis_config, run_analysis
from IPython.display import Image, Markdown, display

REPO_ROOT

In [ ]:
CONFIG_PATH = REPO_ROOT / 'experiments/frustration/history_dependence/analysis_config.yaml'
cfg = load_analysis_config(CONFIG_PATH)
cfg['output']

In [ ]:
artifacts = run_analysis(cfg)
display(Markdown(f"**Output dir:** `{artifacts['output_dir']}`"))
artifacts['runs'].groupby('condition').size().rename('n_runs')

## Main Tables

In [ ]:
display(Markdown('### Effect sizes: free-wall minus free-free'))
artifacts['effect_sizes'].sort_values('observable').reset_index(drop=True)

In [ ]:
if artifacts.get('main_observable'):
    display(Markdown('### Main Pair-Class Superiority Observable'))
    display(Markdown(artifacts['main_observable']['text_summary']))
    import pandas as pd
    pd.DataFrame([{
        'distance_name': artifacts['main_observable']['summary']['distance_name'],
        'mean_free_free': artifacts['main_observable']['summary']['mean_free_free'],
        'mean_free_wall': artifacts['main_observable']['summary']['mean_free_wall'],
        'delta_mean': artifacts['main_observable']['summary']['delta_mean'],
        'ratio_mean': artifacts['main_observable']['summary']['ratio_mean'],
        'A_tie': artifacts['main_observable']['summary']['A_tie'],
        'CI_low': artifacts['main_observable']['summary']['CI_low'],
        'CI_high': artifacts['main_observable']['summary']['CI_high'],
        'pvalue_greater': artifacts['main_observable']['summary']['pvalue_greater'],
    }])
else:
    display(Markdown('Main pair-class superiority observable is unavailable for current inputs.'))

In [ ]:
display(Markdown('### Permutation tests on pairwise distance matrices'))
artifacts['permutation_tests'].sort_values('observable').reset_index(drop=True)

In [ ]:
if not artifacts['run_metrics'].empty:
    display(Markdown('### Run-level MSC and coarse trajectory observables'))
    display(artifacts['run_metrics'].sort_values(['condition', 'run_id']).reset_index(drop=True))
else:
    display(Markdown('Trajectory-based observables are unavailable for this dataset.'))

## Figures

In [ ]:
figure_order = [
    'representative_frames',
    'embedding_distance_matrix',
    'embedding_strip',
    'pair_class_strip',
    'pair_class_cdf',
    'pairwise_distance_matrix',
    'delta_h_examples',
    'delta_h_strip',
    'embedding_vs_delta_h',
    'msc_table',
]

for key in figure_order:
    fig_path = artifacts['figures'].get(key)
    if not fig_path:
        display(Markdown(f'- `{key}`: skipped or unavailable for current inputs.'))
        continue
    display(Markdown(f'### {key}'))
    display(Image(filename=fig_path))

## Saved Artifacts

Полезные файлы после запуска:

- `tables/run_catalog.csv`
- `tables/pairwise_distances.csv`
- `tables/pairwise_summary.csv`
- `tables/effect_sizes.csv`
- `tables/permutation_tests.csv`
- `tables/concordance.csv`
- `tables/run_trajectory_observables.csv`
- `main_observable/tables/main_observable_summary.csv`
- `main_observable/tables/raw_pairwise_values.csv`
- `main_observable/main_observable_report.txt`
- `matrices/*.csv`
- `figures/*.png`